# PQAX footprint of a workspace

Collects the footprint of the items of a Fabric workspace with `pqax collect`, as the identity that runs the notebook, and copies the file into the `Files` area of the default lakehouse of the notebook. It reads OneLake and writes only the footprint. See `templates/fabric/README.md` in https://github.com/pqax-org/pqax.

Attach a default lakehouse, set the parameters, and run all. A pipeline can pass the parameters to the next cell, which is marked as the parameter cell.

In [ ]:
WORKSPACE = ""                                  # the name or the id of the workspace to collect
ITEMS = ""                                      # the items, separated by commas, such as "Sales.Lakehouse, Finance.Warehouse"
PQAX_VERSION = ""                               # the version of Pqax.Cli: the first release after 0.2.0
OUTPUT_FOLDER = "/lakehouse/default/Files/footprints"   # where the footprint goes: the Files area of the default lakehouse
DOTNET_CHANNEL = "8.0"                          # the .NET runtime that the tool runs on: 8.0 or 10.0
EXTRA_OPTIONS = ""                              # more options of pqax collect, separated by spaces, such as "--level 1"

## Install the .NET runtime and the tool

The runtime goes to `~/.dotnet` with the official `dotnet-install.sh`, without the SDK. The tool is the NuGet package `Pqax.Cli`, unpacked: a .NET tool holds its assemblies and their dependencies in `tools/<framework>/any`, so it runs with `dotnet Pqax.Cli.dll` and needs no `dotnet tool install`. Both stay for the session, and a second run skips them.

In [ ]:
import io
import os
import pathlib
import subprocess
import urllib.request
import zipfile

for name, value in (("WORKSPACE", WORKSPACE), ("ITEMS", ITEMS), ("PQAX_VERSION", PQAX_VERSION)):
    if not value.strip():
        raise ValueError(f"Set the parameter {name}.")

home = pathlib.Path.home()
dotnet_root = home / ".dotnet"
dotnet = dotnet_root / "dotnet"
runtime = dotnet_root / "shared" / "Microsoft.NETCore.App"
if not runtime.is_dir() or not any(path.name.startswith(DOTNET_CHANNEL + ".") for path in runtime.iterdir()):
    script = home / "dotnet-install.sh"
    urllib.request.urlretrieve("https://dot.net/v1/dotnet-install.sh", script)
    subprocess.run(["bash", str(script), "--channel", DOTNET_CHANNEL, "--runtime", "dotnet", "--install-dir", str(dotnet_root)], check=True)

tool = home / "pqax" / PQAX_VERSION
if not tool.is_dir():
    url = f"https://api.nuget.org/v3-flatcontainer/pqax.cli/{PQAX_VERSION.lower()}/pqax.cli.{PQAX_VERSION.lower()}.nupkg"
    with urllib.request.urlopen(url) as response:
        package = response.read()
    with zipfile.ZipFile(io.BytesIO(package)) as archive:
        archive.extractall(tool)
cli = tool / "tools" / f"net{DOTNET_CHANNEL}" / "any" / "Pqax.Cli.dll"
if not cli.is_file():
    raise FileNotFoundError(f"{cli} is not in the package: DOTNET_CHANNEL is 8.0 or 10.0.")
environment = dict(os.environ, DOTNET_ROOT=str(dotnet_root), DOTNET_CLI_TELEMETRY_OPTOUT="1", DOTNET_NOLOGO="1")


def pqax(*arguments, env=None):
    """Runs pqax with the arguments, prints what it prints, and returns the exit code."""
    completed = subprocess.run([str(dotnet), str(cli), *arguments], env=env or environment, capture_output=True, text=True)
    print(completed.stdout + completed.stderr)
    return completed.returncode


pqax("--version")

## Collect

`notebookutils.credentials.getToken("storage")` gives a token for `https://storage.azure.com` of the identity that runs the notebook: the user in an interactive session, the owner of the schedule or of the pipeline otherwise. `pqax collect` reads it from an environment variable of its own process, `--access-token-env`, so the token is not on a command line and is not printed. A token lasts about an hour: a collection that lasts longer fails with a 401.

In [ ]:
import datetime
import re

slug = re.sub(r"[^a-z0-9]+", "-", WORKSPACE.lower()).strip("-") or "workspace"
name = f"{slug}-{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d}.pqax"
local = pathlib.Path("/tmp") / name
arguments = ["collect", "--workspace", WORKSPACE, "--output", str(local), "--access-token-env", "PQAX_ACCESS_TOKEN"]
for item in ITEMS.split(","):
    if item.strip():
        arguments += ["--item", item.strip()]
arguments += EXTRA_OPTIONS.split()

token = notebookutils.credentials.getToken("storage")
exit_code = pqax(*arguments, env=dict(environment, PQAX_ACCESS_TOKEN=token))
del token
if exit_code != 0:
    raise RuntimeError(f"pqax collect ended with exit code {exit_code}: 1 for a file that is not valid, 2 for a failure.")

## Check and keep the footprint

In [ ]:
import shutil

if pqax("check", str(local)) != 0:
    raise RuntimeError(f"{name} is not valid.")
pqax("info", str(local))
pqax("tables", str(local), "--top", "20")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
shutil.copy(local, os.path.join(OUTPUT_FOLDER, name))
print(f"{name} is in {OUTPUT_FOLDER}.")